<a href="https://colab.research.google.com/github/CharlesKwak/environment-shirtdaffodil-production/blob/master/LLM_test_SKTGPT2_base_v2_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



#### 초기 GPT 모델 테스트

In [20]:
# 필요 라이브러리 설치
!pip -q install langchain
!pip install ratsnlp

!pip install langchain-community

  Using cached ratsnlp-1.0.53-py3-none-any.whl.metadata (741 bytes)
  Using cached pytorch_lightning-1.6.1-py3-none-any.whl.metadata (33 kB)
Requested pytorch-lightning==1.6.1 from https://files.pythonhosted.org/packages/77/5f/7f0b36c036334cb416230a7909b54c9a128a38623d2e18e87170e2055cd9/pytorch_lightning-1.6.1-py3-none-any.whl (from ratsnlp) has invalid metadata: .* suffix can only be used with `==` or `!=` operators
    torch (>=1.8.*)
           ~~~~~~^
Please use pip<24.1 if you need to use this version.
INFO: pip is looking at multiple versions of ratsnlp to determine which version is compatible with other requirements. This could take a while.
  Using cached ratsnlp-1.0.52-py3-none-any.whl.metadata (760 bytes)
  Using cached pytorch_lightning-1.6.1-py3-none-any.whl.metadata (33 kB)
Requested pytorch-lightning==1.6.1 from https://files.pythonhosted.org/packages/77/5f/7f0b36c036334cb416230a7909b54c9a128a38623d2e18e87170e2055cd9/pytorch_lightning-1.6.1-py3-none-any.whl (from ratsnlp)

In [21]:
from transformers import GPT2LMHeadModel, pipeline
from transformers import PreTrainedTokenizerFast
# from langchain.llms import HuggingFacePipeline
# from langchain import PromptTemplate, LLMChain
import torch

In [22]:
str_model_name = "skt/kogpt2-base-v2"

from transformers import GPT2LMHeadModel
base_model = GPT2LMHeadModel.from_pretrained(
    str_model_name,
)
base_model.eval()


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(51200, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=51200, bias=False)
)

In [23]:
base_model.lm_head

Linear(in_features=768, out_features=51200, bias=False)

In [24]:
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    str_model_name,
    eos_token="</s>",
)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPT2Tokenizer'. 
The class this function is called from is 'PreTrainedTokenizerFast'.


Model Test

In [25]:
# Test Prompt

# test_input = "안녕하세요?"
test_input = "조선시대 최고의 장군은 누구야?"
# test_input = "Who is the best general in Chosun Dynasity?"
input_ids = tokenizer.encode(test_input, return_tensors="pt")



In [26]:
print("input_ids == ", input_ids)

input_ids ==  tensor([[10205, 13222, 36215, 14938,  7991,   406]])


In [27]:
# Greedy Search
#
with torch.no_grad():
    generated_ids = base_model.generate(
        input_ids,
        do_sample=False,
        min_length=10,
        max_length=50,
    )
    print("generated_ids == ", generated_ids)
    print(tokenizer.decode([el.item() for el in generated_ids[0]]))

generated_ids ==  tensor([[10205, 13222, 36215, 14938,  7991,   406, 16563,   377,  6947,  7406,
         16691, 16563,   377,  6947,  7406, 10846,  9022, 36215, 14938,  7991,
           406, 16563,   377,  6947,  7406, 16691, 16563,   377,  6947,  7406,
         16691, 16563,   377,  6947,  7406, 16691, 16563,   377,  6947,  7406,
         16691, 16563,   377,  6947,  7406, 16691, 16563,   377,  6947,  7406]])
조선시대 최고의 장군은 누구야?"
"그렇습니다."
"그렇다면 그 장군은 누구야?"
"그렇습니다."
"그렇습니다."
"그렇습니다."
"그렇습니다."
"그렇습니다."
"그렇


In [28]:
# Beam Search

with torch.no_grad():
    generated_ids = base_model.generate(
        input_ids,
        do_sample=False,
        min_length=10,
        max_length=50,
        num_beams=3,
        no_repeat_ngram_size=3,
    )
    print(tokenizer.decode([el.item() for el in generated_ids[0]]))

조선시대 최고의 장군은 누구야?"
"그렇습니다."
"아니오. 그건 그렇고요."
"그래, 그게 무슨 소리야? 그게 뭐냐고?"
그녀는 고개를 끄덕였다.
"그건 그렇고,
